In [4]:
import datetime
import json

import boto3
import requests

# Assignment: Wikipedia Page Views Pipeline

Build a data pipeline for Wikipedia page views, similar to what we did for edits.

## Overview

You will:
1. Create a notebook that extracts top viewed Wikipedia pages
2. Create Athena tables to query the data
3. Deploy as a Lambda function
4. Schedule with EventBridge

**Use the same `<username>` you used for the edits pipeline!**

## Part 1: Create the Notebook

Create a new notebook in `/homework/extract_views.ipynb` that extracts, transforms, and loads page view data.


In [8]:
# Try different dates to see how the data changes
DATE_PARAM = "2025-10-02"

# best practice: as soon as you have a date string, convert it to a date object!
date = datetime.datetime.strptime(DATE_PARAM, "%Y-%m-%d")
date

datetime.datetime(2025, 10, 2, 0, 0)

In [6]:
date.strftime("%Y/%m/%d")

'2025/10/02'

In [ ]:
# use the same bucket and the same username
# we don't need the bucket creation part again

# /pageviews/top/en.wikipedia.org/all-access/date.strftime("%Y/%m/%d")

url = f"https://wikimedia.org/api/rest_v1/metrics/pageviews/top/en.wikipedia.org/all-access/{date.strftime("%Y/%m/%d")}"

print(f"Requesting REST API URL: {url}")

# Make the API request
wiki_server_response = requests.get(url, headers={"User-Agent": "curl/7.68.0"}) # wikipedia doesn't have an anti-crawler policy but you need to specify who you are
wiki_response_status = wiki_server_response.status_code # must be 200 for http OK
wiki_response_body = wiki_server_response.text

print(f"Wikipedia REST API Response body: {wiki_response_body[:500]}...")
print(f"Wikipedia REST API Response Code: {wiki_response_status}")

# Validate response
if wiki_response_status != 200:
    raise Exception(f"Received non-OK status code from Wiki Server: {wiki_response_status}") # will stop execution
print(f"Successfully retrieved Wikipedia data, content-length: {len(wiki_response_body)}")

Requesting REST API URL: https://wikimedia.org/api/rest_v1/metrics/pageviews/top/en.wikipedia.org/all-access/2025/10/02
Wikipedia REST API Response body: {"items":[{"project":"en.wikipedia","access":"all-access","year":"2025","month":"10","day":"02","articles":[{"article":"Main_Page","views":6040119,"rank":1},{"article":"Jane_Goodall","views":954670,"rank":2},{"article":"Special:Search","views":889727,"rank":3},{"article":".xxx","views":633935,"rank":4},{"article":"Yom_Kippur","views":412350,"rank":5},{"article":"Google_Chrome","views":406871,"rank":6},{"article":"Kantara:_Chapter_1","views":304128,"rank":7},{"article":"Wikipedia:Featured_picture...
Wikipedia REST API Response Code: 200
Successfully retrieved Wikipedia data, content-length: 56211


In [ ]:
### STILL ADAPT THIS!

# Parse the API response and extract top edits
wiki_response_parsed = wiki_server_response.json()
# we need to figure out "items", "results" etc.; look at the JSON structure; also on wikimedia API website
top_edits = wiki_response_parsed["items"][0]["results"][0]["top"]

# Transform to JSON Lines format
# modern way for Athena to work with JSON files; can stack multiple JSON objects in a file, one per line
current_time = datetime.datetime.now(datetime.timezone.utc)
json_lines = ""
for page in top_edits[:5]: # taking the first five edits only
    record = {
        "title": page["page_title"], # keep title
        "edits": page["edits"], # keep number of edits
        "date": date.strftime("%Y-%m-%d"), # date that we passed
        "retrieved_at": current_time.replace(tzinfo=None).isoformat(), # best practice when working with warehouses
    }
    json_lines += json.dumps(record) + "\n"

print(f"Transformed {len(top_edits)} records to JSON Lines")
print(f"First few lines{json_lines[:500]}...")


**API Documentation:** https://doc.wikimedia.org/generated-data-platform/aqs/analytics-api/reference/page-views.html

Look for the endpoint that returns the **most-viewed pages** for a given day.

Your notebook should:
1. Fetch the top viewed pages for a specific date
2. Transform the response to JSON Lines format with these fields:
   - `title` - the article title
   - `views` - the view count
   - `rank` - the ranking position
   - `date` - the date being queried
   - `retrieved_at` - timestamp when you fetched the data
3. Upload to S3 at `raw-views/raw-views-YYYY-MM-DD.json`
4. Re-execute the notebook twice more, setting the date to different days

Use the same bucket as edits: `<username>-wikidata`

**Tip:** The API response structure is slightly different from the edits API. Explore the response JSON to find where the article data is located.

## Part 2: Create Athena Tables

Create two SQL files:

1. `4_raw_views.sql` - External table called `raw_views` pointing to your S3 data
2. `5_views_view.sql` - View called `views` with proper timestamp casting + **records ordered by `date` (primary, ascending) and `rank` (secondary, ascending)**.

Look at `2_raw_edits.sql` and `3_edits_view.sql` for reference. Your table columns should match the JSON fields from Part 1.

## Part 3: Create Lambda Function

1. Create `lambda_extract_views.py` (convert the logic of your notebook as we did with _Wikipedia Edits_)
   - The Lambda should default to fetching data from 21 days ago if no date is provided
   - Accept an optional `{"date": "YYYY-MM-DD"}` parameter to fetch a specific date

2. Deploy to AWS Lambda
   - **Function name:** `WikiViewsLambda<Username>` (e.g., `WikiViewsLambdaJohndoe`)
   - **Runtime:** Python 3.13
   - Once you created the function, add the `AWSSDKPandas-Python313` layer (version 5, from AWS layers)
   - In _Configuration_ -> _Permissions_, edit the _Execution Role_ and attach the `LambdaS3ExecutionRole` role (this is a role with S3 write access)
   - In _General Configuration_, set timeout to 30 seconds

3. Test both with `{"date": "2025-11-20"}` and with `{}` (empty event) - this will use the default date (21 days ago)

## Part 4: Schedule with EventBridge

1. Go to Amazon EventBridge → Schedules → Create schedule
2. Configure:
   - **Schedule name:** `WikiViewsScheduler<Username>` (e.g., `WikiViewsSchedulerJohndoe`)
   - **Schedule type:** Recurring schedule
   - **Cron expression:** `10 0 * * ? *` (daily at 0:10 AM UTC)
   - **Flexible time window:** 1 hour (cost saving)
   - **Target:** AWS Lambda → your Lambda function
   - **Payload:** `{}`
   - **Role:** Use `EventBridgeLambdaRole`

## Deliverables

In your `/homework` folder on GitHub:
1. `extract_views.ipynb` - the notebook
2. `lambda_extract_views.py` - the Lambda code
3. `4_raw_views.sql` - table creation SQL
4. `5_views_view.sql` - view creation SQL

## Grading Criteria
Ensure that all assets you submit check all the requirements above:
1) Homework in the `homework` folder on Git, committed and pushed to GitHub
2) You use the same username, bucket, and database as in your in-class exercise
3) Notebook in place and checks every requirement above
4) SQL files in place and check all requirements above
5) Lambda function in place and checks all requirements above
6) Lambda function deployed under the function name required and works both with and without passing a date parameter
7) EventBridge created with the schedule name required

Deadline: 17 Dec 2025, 23:59. Same terms apply as with the previous deadlines

Submission URL on Moodle